# 🎤 AI Vocals Studio — Voice Model Training

**Before running:** `Runtime` → `Change runtime type` → **T4 GPU** → Save

Then click **Runtime → Run all** and follow the prompts.

**Training saves checkpoints every 100 epochs and backs up to Google Drive every 30 minutes.**

In [ ]:
# ── Step 1: Check GPU ────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'✅ GPU ready: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('❌ No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Step 2: Install dependencies ─────────────────────────────────
!pip install -q so-vits-svc-fork==4.2.30 torchcodec
print('✅ Dependencies installed')

In [ ]:
# ── Step 3: Mount Google Drive ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create backup directory
import os
BACKUP_DIR = '/content/drive/MyDrive/ai_vocals_checkpoints'
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f'✅ Drive mounted. Checkpoints will backup to: {BACKUP_DIR}')

In [ ]:
# ── Step 4: Upload and extract training data ─────────────────────
import glob, os, zipfile

# Option A: Auto-find zip in Drive
search_patterns = [
    '/content/drive/MyDrive/**/*training*.zip',
    '/content/drive/MyDrive/**/*.zip',
]

zip_path = None
for pat in search_patterns:
    found = glob.glob(pat, recursive=True)
    if found:
        # Get most recent
        zip_path = max(found, key=os.path.getmtime)
        print(f'Found: {zip_path}')
        break

# Option B: Manual upload if not found in Drive
if not zip_path:
    print('No zip found in Drive. Upload manually:')
    from google.colab import files
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]

# Extract
print(f'📦 Extracting: {os.path.basename(zip_path)}')
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/')
print('✅ Extracted')

In [ ]:
# ── Step 5: Verify dataset ───────────────────────────────────────
import glob as g
features = g.glob('/content/dataset/44k/**/*.data.pt', recursive=True)
configs = g.glob('/content/configs/44k/config.json')

print(f'Feature files: {len(features)}')
print(f'Config found: {bool(configs)}')

if not features:
    raise RuntimeError('No .data.pt files found')
if not configs:
    raise RuntimeError('No config.json found')

print('✅ Dataset verified')

In [ ]:
# ── Step 6: Resume from checkpoint if exists ─────────────────────
import shutil

# Check if we have checkpoints in Drive from previous run
existing = g.glob(f'{BACKUP_DIR}/G_*.pth')
if existing:
    # Sort by epoch number
    latest = sorted(existing, key=lambda p: int(''.join(filter(str.isdigit, p))) or 0)[-1]
    print(f'🔄 Resuming from: {os.path.basename(latest)}')
    shutil.copy(latest, '/content/logs/44k/')
    # Also copy D_ checkpoint
    d_ckpt = latest.replace('G_', 'D_')
    if os.path.exists(d_ckpt):
        shutil.copy(d_ckpt, '/content/logs/44k/')
else:
    print('🆕 Starting fresh training')

In [ ]:
# ── Step 7: Setup auto-backup to Drive ───────────────────────────
import threading
import time

def backup_to_drive():
    """Backup checkpoints to Drive every 30 minutes"""
    while True:
        time.sleep(1800)  # 30 minutes
        ckpts = g.glob('/content/logs/44k/G_*.pth')
        if ckpts:
            for ckpt in ckpts:
                dst = os.path.join(BACKUP_DIR, os.path.basename(ckpt))
                shutil.copy(ckpt, dst)
            print(f'💾 Backed up {len(ckpts)} checkpoints to Drive')

# Start backup thread
backup_thread = threading.Thread(target=backup_to_drive, daemon=True)
backup_thread.start()
print('✅ Auto-backup enabled (every 30 min)')

In [ ]:
# ── Step 8: TRAIN ────────────────────────────────────────────────
import os
os.chdir('/content')

print('🚀 Starting training...')
print('⏱️  This takes 6-12 hours')
print('💾 Checkpoints saved every 100 epochs')
print('💾 Backed up to Drive every 30 minutes')
print('✅ Colab will keep running if you close the browser')
print('')

!svc train -c configs/44k/config.json -m logs/44k

print('✅ Training complete!')

In [ ]:
# ── Step 9: Final backup and download ────────────────────────────
import glob, os, shutil
from google.colab import files

# Find all checkpoints
ckpts = sorted(glob.glob('/content/logs/44k/G_*.pth'),
               key=lambda p: int(''.join(filter(str.isdigit, p))) or 0)

print('All checkpoints:', [os.path.basename(c) for c in ckpts])

if ckpts:
    best = ckpts[-1]
    
    # Final backup to Drive
    for ckpt in ckpts:
        shutil.copy(ckpt, BACKUP_DIR)
    shutil.copy('/content/logs/44k/config.json', BACKUP_DIR)
    print(f'✅ All checkpoints backed up to: {BACKUP_DIR}')
    
    # Download the best one
    print(f'\n📥 Downloading: {os.path.basename(best)}')
    files.download(best)
    files.download('/content/logs/44k/config.json')
else:
    print('⚠️ No checkpoints found')

In [ ]:
# ── Step 10: Install to local machine ────────────────────────────
# After downloading, run this on your local machine:
# mkdir -p models/YOUR_MODEL_NAME
# mv ~/Downloads/G_*.pth models/YOUR_MODEL_NAME/model.pth
# mv ~/Downloads/config.json models/YOUR_MODEL_NAME/

print('🎉 DONE!')
print('')
print('Next steps:')
print('1. Move downloaded files to: models/<model_name>/')
print('2. Open AI Vocals Studio')
print('3. Select your model and generate!')